In [ ]:
import json
import random
import re

INPUT_FILE = "/content/smartserve_final_dataset_improved(3) (1).json"
OUTPUT_FILE = "smartserve_final_dataset.json"

MIN_SAMPLES = 70


GENERAL_CONVERSATION = {
    "Greeting","Farewell","ThankYou","not_happy","unknown"
}

APP_NAVIGATION = {
    "login","sign_up","home_screen","drawer",
    "my_account_profile","request_service_screen",
    "post_service_screen","rating_review",
    "internal_chatbot","external_chatbot","certified"
}

MARKETPLACE_INTERACTION = {
    "provider_search","availability_status","contact_provider",
    "service_search_general","service_categories",
    "service_location_query","service_time_query",
    "service_pricing","pricing_cost","payment_method_query",
    "urgent_service_request"
}

SERVICE_CLASSIFICATION = {
    "house_cleaning","electrician","plumber","carpenter","barber",
    "automotive","pest_control","lawn_and_garden","health_wellness",
    "child_elder_care","junk_removal","home_shifting","pc_installation",
    "beauty_and_spa","painting_and_decorating","appliance_repair",
    "handyman","pet_services","photography","tutor","interior_design",
    "exterior_design","decorating_services","party_planning",
    "makeup_artist","event_management","catering_services",
    "event_decoration","photography_videography","event_staffing",
    "fireworks_display","dj_entertainment","repair_restoration",
    "mobile_phone_repair","laptop_computer_repair",
    "carpet_upholstery_cleaning","window_cleaning","pool_maintenance",
    "moving_services","storage_services","home_automation",
    "solar_panel_installation","hvac_services","furniture_assembly",
    "custom_furniture","furniture_repair","upholstery_services",
    "furniture_polishing","furniture_cleaning","furniture_moving",
    "office_furniture","printing_services","commercial_printing",
    "digital_printing","large_format_printing","3d_printing",
    "photocopy_scanning","binding_laminating","graphic_design_print"
}




def assign_group(intent):

    if intent in GENERAL_CONVERSATION:
        return "general_conversation"

    if intent in APP_NAVIGATION:
        return "app_navigation"

    if intent in MARKETPLACE_INTERACTION:
        return "marketplace_interaction"

    if intent in SERVICE_CLASSIFICATION:
        return "service_classification"

    return "other"




def normalize(text):

    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text




def remove_greeting_noise(text):

    greetings = ["hi","hello","hey"]

    words = text.split()

    if len(words) > 3 and words[0] in greetings:
        words = words[1:]

    return " ".join(words)




def remove_cross_intent_duplicates(intents):

    seen = {}
    cleaned = []

    for intent in intents:

        unique_texts = []

        for text in intent["text"]:

            t = normalize(text)

            if t not in seen:

                seen[t] = intent["intent"]
                unique_texts.append(t)

        intent["text"] = unique_texts
        cleaned.append(intent)

    return cleaned




synonyms = {
    "need": ["require","looking for","want"],
    "fix": ["repair","solve"],
    "install": ["setup","fit"],
    "find": ["search for","look for"],
    "help": ["assist","support"],
    "show": ["display","open"],
    "open": ["launch","access"],
    "post": ["publish","create"]
}


def augment(sentence):

    words = sentence.split()

    for i,w in enumerate(words):

        if w in synonyms:
            words[i] = random.choice(synonyms[w])

    return " ".join(words)




def balance(texts):

    texts = list(set(texts))

    attempts = 0
    max_attempts = MIN_SAMPLES * 20

    while len(texts) < MIN_SAMPLES and attempts < max_attempts:

        s = random.choice(texts)

        new_s = augment(s)
        new_s = normalize(new_s)

        if new_s not in texts and len(new_s.split()) >= 2:
            texts.append(new_s)

        attempts += 1

    return texts




def process_dataset(data):

    intents = data["intents"]

    intents = remove_cross_intent_duplicates(intents)

    processed = []

    for intent in intents:

        texts = []

        for t in intent["text"]:

            t = normalize(t)
            t = remove_greeting_noise(t)

            if len(t.split()) >= 2:
                texts.append(t)

        texts = list(set(texts))

        if len(texts) < MIN_SAMPLES:
            texts = balance(texts)

        processed.append({

            "intent": intent["intent"],
            "group": assign_group(intent["intent"]),
            "description": intent.get("description",""),
            "text": texts,
            "responses": intent["responses"]

        })

    return {"intents": processed}



def main():

    with open(INPUT_FILE,"r",encoding="utf8") as f:
        data = json.load(f)

    data = process_dataset(data)

    with open(OUTPUT_FILE,"w",encoding="utf8") as f:
        json.dump(data,f,indent=2)

    print("SmartServeAI dataset preprocessing complete")
    print("Saved to:", OUTPUT_FILE)


if __name__ == "__main__":
    main()

SmartServeAI dataset preprocessing complete
Saved to: smartserve_final_dataset.json


In [ ]:
import json
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)



DATA_PATH = "/content/smartserve_final_dataset.json"

with open(DATA_PATH) as f:
    dataset = json.load(f)

texts = []
labels = []

for intent in dataset["intents"]:
    for t in intent["text"]:
        texts.append(t)
        labels.append(intent["intent"])

print("Total samples:", len(texts))




label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)

num_labels = len(label_encoder.classes_)

print("Total intents:", num_labels)



train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels_encoded,
    test_size=0.15,
    random_state=42,
    stratify=labels_encoded
)




tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=64
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=64
)


class IntentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


train_dataset = IntentDataset(train_encodings, train_labels)
val_dataset = IntentDataset(val_encodings, val_labels)




model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)




training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=10,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=50,

    save_strategy="no"
)




trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset
)




trainer.train()




predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(val_labels, preds)

print("\nValidation Accuracy:", accuracy)

Total samples: 7078
Total intents: 86


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,4.448791
100,4.355424
150,4.212323
200,4.037284
250,3.875046
300,3.696195
350,3.526036
400,3.285117
450,3.133555
500,2.937193



Validation Accuracy: 0.827683615819209


In [ ]:
for intent_data in data["intents"]:
    intent_name = intent_data["intent"]
    num_examples = len(intent_data["text"])
    print(f"Intent: {intent_name}, Examples: {num_examples}")

Intent: Greeting, Examples: 90
Intent: Farewell, Examples: 89
Intent: ThankYou, Examples: 90
Intent: not_happy, Examples: 90
Intent: login, Examples: 90
Intent: sign_up, Examples: 89
Intent: home_screen, Examples: 87
Intent: internal_chatbot, Examples: 87
Intent: external_chatbot, Examples: 90
Intent: my_account_profile, Examples: 87
Intent: request_service_screen, Examples: 90
Intent: post_service_screen, Examples: 90
Intent: drawer, Examples: 72
Intent: rating_review, Examples: 90
Intent: certified, Examples: 89
Intent: house_cleaning, Examples: 90
Intent: electrician, Examples: 90
Intent: plumber, Examples: 90
Intent: carpenter, Examples: 90
Intent: barber, Examples: 90
Intent: automotive, Examples: 90
Intent: pest_control, Examples: 90
Intent: lawn_and_garden, Examples: 90
Intent: health_wellness, Examples: 90
Intent: child_elder_care, Examples: 90
Intent: junk_removal, Examples: 90
Intent: home_shifting, Examples: 90
Intent: pc_installation, Examples: 90
Intent: beauty_and_spa, Ex

In [ ]:
#Accuracy

predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(val_labels, preds))

Accuracy: 0.827683615819209


In [ ]:
import pickle
import os

# ----------------------------
# Create folder
# ----------------------------

save_path = "/content/chatbot_model"
os.makedirs(save_path, exist_ok=True)

# ----------------------------
# Save model and tokenizer
# ----------------------------

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# ----------------------------
# Save label encoder
# ----------------------------

with open(f"{save_path}/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# ----------------------------
# Save responses
# ----------------------------

responses = {}

for intent in dataset["intents"]:
    responses[intent["intent"]] = intent["responses"]

with open(f"{save_path}/responses.pkl", "wb") as f:
    pickle.dump(responses, f)

print("\nModel saved successfully")
print("Saved at:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully
Saved at: /content/chatbot_model


In [ ]:
import shutil

# Folder to zip
folder_path = "/content/chatbot_model"

# Output zip file
zip_path = "/content/chatbot_model.zip"

# Create zip
shutil.make_archive(zip_path.replace(".zip",""), 'zip', folder_path)

print("ZIP file created successfully:", zip_path)

ZIP file created successfully: /content/chatbot_model.zip


In [ ]:
import torch
import pickle
import random
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast



model_path = "/content/chatbot_model"

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)

label_encoder = pickle.load(open(f"{model_path}/label_encoder.pkl","rb"))
responses = pickle.load(open(f"{model_path}/responses.pkl","rb"))

model.eval()

previous_intent = None




def predict_intent(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    confidence, predicted_class = torch.max(probs, dim=1)

    intent = label_encoder.inverse_transform([predicted_class.item()])[0]

    return intent, confidence.item()




print("\nChatbot Ready (type 0 to stop)\n")

while True:

    user = input("You: ")

    if user == "0":
        break

    intent, conf = predict_intent(user)

    global previous_intent

    print("Intent:", intent)
    print("Confidence:", round(conf,3))


    if conf < 0.40:
        print("Bot: I couldn't understand, can you please explain it clearly?\n Like provider or seeker and which type of service catagory you belong")
        continue


    if intent in ["service_pricing","service_time_query","service_location_query"]:

        if previous_intent and previous_intent in responses:

            print("Bot:", random.choice(responses[intent]))
            print("(Related to previous service:", previous_intent,")\n")

        else:
            print("Bot: Please tell me which service you are referring to.\n")

    else:

        if intent in responses:
            print("Bot:", random.choice(responses[intent]),"\n")
        else:
            print("Bot: I couldn't understand your request clearly.\n")


    previous_intent = intent

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Chatbot Ready (type 0 to stop)

You: hi
Intent: Greeting
Confidence: 0.949
Bot: Hello there! Ready to assist with whatever you need. How can I help? 

You: want to got service
Intent: request_service_screen
Confidence: 0.711
Bot: Once posted, nearby professionals will see your request and can respond. 

You: want to request for pest control
Intent: pest_control
Confidence: 0.973
Bot: All our pest control providers are licensed and follow safety regulations. 

You: how to request this service?
Intent: request_service_screen
Confidence: 0.878
Bot: To request a service, go to the home screen and select the category you need, or tap on 'Request Service' to describe your job in detail. 

You: ok
Intent: Farewell
Confidence: 0.789
Bot: Bye for now! Wishing you success with your service. 

You: thanks
Intent: ThankYou
Confidence: 0.959
Bot: Glad I could help! Feel free to ask if you have more questions. 

You: 0


In [ ]:
import json
from collections import defaultdict

DATASET_PATH = "/content/smartserve_final_dataset_improved.json"

with open(DATASET_PATH,"r",encoding="utf8") as f:
    dataset = json.load(f)

intents = dataset["intents"]

print("\n========= DATASET DIAGNOSTIC =========\n")



print("Total intents:", len(intents))




print("\nSamples per intent:\n")

intent_sizes = {}

for intent in intents:
    count = len(intent["text"])
    intent_sizes[intent["intent"]] = count
    print(intent["intent"], ":", count)


min_samples = min(intent_sizes.values())
max_samples = max(intent_sizes.values())

print("\nMin samples:", min_samples)
print("Max samples:", max_samples)




print("\nChecking duplicate sentences across intents...")

sentence_map = defaultdict(list)

for intent in intents:
    for s in intent["text"]:
        sentence_map[s].append(intent["intent"])

duplicates = {k:v for k,v in sentence_map.items() if len(v)>1}

print("Duplicate sentences found:", len(duplicates))

if len(duplicates)>0:
    print("\nExample duplicates:\n")
    for i,(sent,intent_list) in enumerate(duplicates.items()):
        print(sent,"->",intent_list)
        if i==10:
            break



print("\nChecking overlapping vocabulary...")

intent_words = {}

for intent in intents:
    words = set()
    for t in intent["text"]:
        words.update(t.split())
    intent_words[intent["intent"]] = words

overlap_report = []

intent_names = list(intent_words.keys())

for i in range(len(intent_names)):
    for j in range(i+1,len(intent_names)):

        a = intent_names[i]
        b = intent_names[j]

        inter = intent_words[a].intersection(intent_words[b])

        if len(inter)>10:

            overlap_report.append((a,b,len(inter)))

print("Potential overlapping intents:",len(overlap_report))

for r in overlap_report[:10]:
    print(r)



print("\nChecking short queries...")

short_queries = []

for intent in intents:
    for t in intent["text"]:
        if len(t.split()) < 2:
            short_queries.append((intent["intent"],t))

print("Short queries found:",len(short_queries))



print("\n========= DATASET HEALTH =========")

if min_samples >= 30:
    print("✔ Balanced dataset")

else:
    print("⚠ Some intents have too few samples")

if len(duplicates) == 0:
    print("✔ No duplicate sentences")

else:
    print("⚠ Duplicate examples exist")

if len(overlap_report) < 5:
    print("✔ Intents are well separated")

else:
    print("⚠ Some intents overlap too much")

print("\nDiagnostic complete\n")



========= DATASET DIAGNOSTIC =========

Total intents: 86

Samples per intent:

Greeting : 90
Farewell : 89
ThankYou : 90
not_happy : 90
login : 90
sign_up : 89
home_screen : 87
internal_chatbot : 87
external_chatbot : 90
my_account_profile : 87
request_service_screen : 90
post_service_screen : 90
drawer : 72
rating_review : 90
certified : 89
house_cleaning : 90
electrician : 90
plumber : 90
carpenter : 90
barber : 90
automotive : 90
pest_control : 90
lawn_and_garden : 90
health_wellness : 90
child_elder_care : 90
junk_removal : 90
home_shifting : 90
pc_installation : 90
beauty_and_spa : 90
painting_and_decorating : 90
appliance_repair : 89
handyman : 90
pet_services : 90
photography : 90
tutor : 90
interior_design : 90
exterior_design : 90
decorating_services : 90
party_planning : 90
makeup_artist : 90
event_management : 90
catering_services : 90
event_decoration : 90
photography_videography : 88
event_staffing : 90
fireworks_display : 90
dj_entertainment : 90
repair_restoration : 86